# Calculations and demos of the stereographic projection

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from scipy.optimize import brentq

In [ ]:
import logging

log = logging.getLogger(__name__)
logging.basicConfig(level=logging.INFO)

In [ ]:
def _invert_one(yi, *, xtol=1e-06):
    if not (0.0 <= yi <= np.pi):
        raise ValueError(f"y must be in [0, Pi], got y={yi}")
    return brentq(lambda x: x - np.sin(x) - yi, a=0.0, b=np.pi, xtol=xtol)
def invert_x_minus_sin_x(y, xtol=1e-06):
    return np.vectorize(_invert_one, otypes=[float])(y, xtol=xtol)

In [ ]:
x_g = np.linspace(0, np.pi, 200)
y = np.linspace(0, np.pi, 100)

In [ ]:
%%time
x_p = invert_x_minus_sin_x(y, xtol=1e-06)

In [ ]:
fig, ax = plt.subplots(figsize=(4, 4), dpi=120)

ax.set_box_aspect(1)
ax.plot(x_g, x_g-np.sin(x_g), label='x - sin(x)',
        color='black', lw=3, zorder=0)
ax.scatter(x_p, y, s=7**2, label='brentq', 
           color='tab:red', marker='x', alpha=0.5, zorder=1)
ax.legend()

plt.show()

## Spherical binner

$$
    \omega(\mathbf{r})
    =
    2 \arctan\left( \frac{\left| \mathbf{r}_{\mathrm{3D}} \right|}{d_{\mathrm{4D}}} \right)
$$

In [ ]:
from abc import ABC, abstractmethod

In [ ]:
class SphericalBinner(ABC):
    '''TODO
    '''
    @abstractmethod
    def r_limit(self, i):
        '''TODO'''
        raise NotImplementedError

    def r_centroid(self, i):
        r_0 = self.r_limit(i)
        r_1 = self.r_limit(i+1)
        return self.centroid(r_0, r_1)

    @staticmethod
    def centroid(r_0, r_1):
        r'''
        Calculates the centroid of a conical frustum along the radius in
        Euclidean space for a bin between ``r_0`` and ``r_1``.
        The formula used is:
        .. math::
            r_i = \frac{1}{4} \frac{r_1^3 - r_0^3}{r_1^2 - r_0^2} + r_0
        
        where :math:`r_0` and :math:`r_1` are the lower and upper limits
        of the bin.

        Parameters
        ----------
        r_0 : float or ndarray
            The lower limit of the bin.
        r_1 : float or ndarray
            The upper limit of the bin.

        Returns
        -------
        c_i : float or ndarray
            The centroid of the bin.

        Notes
        -----
        The formula is defined in a way to avoid numerical instability
        when ``r_0`` and ``r_1`` are very close to each other.
        '''
        nom = (r_1-r_0) * (r_0*r_0 + 2*r_0*r_1 + 3*r_1*r_1)  # r_1^3 - r_0^3
        den = (r_0*r_0 + r_0*r_1 + r_1*r_1)                  # r_1^2 - r_0^2
        return 0.25 * nom / den + r_0

class SphericalConstantOmega(SphericalBinner):
    r'''
    Radial binner that places equally spaced cuts in the stereographic
    polar half-angle :math:`\omega`.

    The first ``n_bins`` shells have an identical angular width
    :math:`\Delta\omega`. An optional fractional extension
    ``last_cell_size`` lets you append an extra (finite) zone so that
    the final grid point lies comfortably short of the projected
    "horizon" at :math:`\omega=\pi/2`.

    Parameters
    ----------
    d_4d : float
        The diameter of the 4D sphere.
    n_bins : int
        The number of radial bins to be used.
    last_cell_size : float
        The size of the last non-infinite cell.
    '''
    def __init__(self, d_4d, n_bins, last_cell_size):
        self.d_4d = d_4d
        self.n_bins = n_bins
        self.last_cell_size = last_cell_size
        self.bin_width = np.pi / (2 * (self.n_bins + self.last_cell_size))

    def r_limit(self, i):
        omega = i * self.bin_width
        return self.d_4d * np.tan(omega/2)

class SphericalConstantVolume(SphericalBinner):
    r'''
    Uniformly spaced shells in the Euclidean space with each shell
    enclosing the same Euclidean volume.

    The total volume inside the half-angle :math:`\omega` is given by
    .. math::
        V(\omega)
        =
        \int_0^{\omega} 4 \pi (d_4d \tan(\omega'))^2 \mathrm{d} (d_4d \tan(\omega'))
        =
        2 \pi d_4d^3 [ x - \sin(x) ]\,, \qquad x \equiv 2 \omega\,.

    Parameters
    ----------
    r_3d : float
        The radius of the simulation volume in the Euclidean space.
    d_4d : float
        The diameter of the 4D hypersphere.
    n_bins : int
        The number of radial bins to be used.
    '''
    def __init__(self, r_3d, d_4d, n_bins):
        self.d_4d = d_4d
        self.n_bins = n_bins
        # Largest possible half-angle, given by r = d_4d * tan(omega/2)
        self.omega_max = 2 * np.arctan(r_3d/d_4d)
        # Radial bin width in the Euclidean space, given by x = 2 * omega
        self.bin_width = (2*self.omega_max - np.sin(2*self.omega_max)) / n_bins

    def r_limit(self, i):
        if i < 0 or i > self.n_bins:
            raise IndexError('Index `i` must be between 0 and `n_bins`.')
        omega = self.invert_x_minus_sin_x(i * self.bin_width) / 2  # TODO: Fix
        return self.d_4d * np.tan(omega/2)

    @staticmethod
    def _invert_one(yi, *, xtol=1e-06):
        '''TODO'''
        if not (0.0 <= yi <= np.pi):
            raise ValueError(f'y must be in [0, Pi], got y={yi}')
        return brentq(lambda x: x - np.sin(x) - yi, a=0.0, b=np.pi, xtol=xtol)
    
    def invert_x_minus_sin_x(self, y, xtol=1e-06):
        r'''
        Inverts the function :math:`y = x - \sin(x)` using Brent's method.
        The function is defined in the range :math:`[0, \pi]`.

        Parameters
        ----------
        y : float or ndarray
            The value to be inverted.
        xtol : float
            The tolerance for the root-finding algorithm.
        
        Returns
        -------
        x : float or ndarray
            The inverted value(s) of :math:`x`.
        '''
        return np.vectorize(self._invert_one, otypes=[float])(y, xtol=xtol)

In [ ]:
r_3d = 1000  # Radius of the simulation in Euclidean space [Mpc]
d_4d = 105   # Diameter of the simulation in 4D space [Mpc]
n_bins = 224

In [ ]:
last_cell_size = n_bins*np.pi / (2*np.arctan(r_3d/d_4d)) - n_bins
log.info(f'last_cell_size = {last_cell_size}')

binner_comg = SphericalConstantOmega(d_4d=d_4d, n_bins=n_bins, last_cell_size=last_cell_size)
log.info(f'binner_comg.bin_width = {binner_comg.bin_width}')
log.info(f'binner_comg.bin_width * n_bins = {binner_comg.bin_width * n_bins}')
binner_cvol = SphericalConstantVolume(r_3d=r_3d, d_4d=d_4d, n_bins=n_bins)
log.info(f'binner_cvol.omega_max = {binner_cvol.omega_max}')
log.info(f'binner_cvol.bin_width = {binner_cvol.bin_width}')
log.info(f'binner_cvol.bin_width * n_bins = {binner_cvol.bin_width * n_bins}')

In [ ]:
fig, ax = plt.subplots(figsize=(4, 4), dpi=120)

i = np.arange(0, n_bins)

ax.plot(i, binner_comg.r_limit(i=i), label='SphericalConstantOmega')
ax.plot(i, binner_comg.r_centroid(i=i), label='SphericalConstantOmega centroid')

ax.plot(i, binner_cvol.r_limit(i=i), label='SphericalConstantVolume')
ax.plot(i, binner_cvol.r_centroid(i=i), label='SphericalConstantVolume centroid')

ax.set_xlabel('Bin index')
ax.legend(loc='upper left', fontsize=8)

plt.show()

## Cylindrical binner

In [ ]:
class CylindricalBinner(ABC):
    '''
    Abstract Base Class for creating radial bins in a cylindrical geometry.
    '''
    @abstractmethod
    def r_limit(self, i):
        '''
        Calculates the radial limit of the i-th bin boundary.
        The 0-th limit is the center (r=0) and the ``n_bins``-th limit
        is the maximum radius of the cylinder.
        '''
        raise NotImplementedError

    def r_centroid(self, i):
        '''Calculates the centroid of the i-th bin.'''
        r_0 = self.r_limit(i)
        r_1 = self.r_limit(i+1)
        return self.centroid(r_0, r_1)

    @staticmethod
    def centroid(r_0, r_1):
        r'''
        Calculates the radial centroid of a cylindrical shell (an annulus)
        between inner radius r_0 and outer radius r_1.

        The formula used is:
        .. math::
            r_c = \frac{2}{3} \frac{r_1^3 - r_0^3}{r_1^2 - r_0^2}
            = \frac{2}{3} \frac{r_0^2 + r_0 r_1 + r_1^2}{r_0 + r_1}

        Parameters
        ----------
        r_0 : float or ndarray
            The inner radius of the shell.
        r_1 : float or ndarray
            The outer radius of the shell.

        Returns
        -------
        c_i : float or ndarray
            The radial centroid of the shell.
        '''
        # Handle the case where r_0 and r_1 are the same or r_0=r_1=0
        if np.allclose(r_0, r_1):
            return r_0
        nom = r_0**2 + r_0 * r_1 + r_1**2
        den = r_0 + r_1
        return (2/3) * nom / den

class CylindricalConstantRadius(CylindricalBinner):
    '''
    Radial binner that creates cylindrical shells of equal radial width.

    Parameters
    ----------
    r_3d : float
        The maximum radius of the cylinder.
    n_bins : int
        The number of radial bins to be used.
    '''
    def __init__(self, r_3d, n_bins):
        if r_3d <= 0:
            raise ValueError('Radius of cylinder `r_3d` must be positive.')
        if n_bins <= 0:
            raise ValueError('Number of bins `n_bins` must be positive.')
        self.r_3d = r_3d
        self.n_bins = n_bins
        self.bin_width = self.r_3d / self.n_bins

    def r_limit(self, i):
        '''Calculates the limit for bins of constant radial width.'''
        return i * self.bin_width

class CylindricalConstantVolume(CylindricalBinner):
    r'''
    Radial binner that creates cylindrical shells with equal volume.

    The volume of a cylinder is V = \pi r^2 H. To ensure each shell
    has the same volume, the radius of the i-th bin limit must scale
    such that V(r_i) is proportional to i. This gives the relation:
    .. math::
        r_i = r_{\text{max}} \sqrt{\frac{i}{N_{\text{bins}}}}

    Parameters
    ----------
    r_3d : float
        The maximum radius of the cylinder.
    n_bins : int
        The number of radial bins to be used.
    '''
    def __init__(self, r_3d, n_bins):
        if r_3d <= 0:
            raise ValueError('Radius of cylinder `r_3d` must be positive.')
        if n_bins <= 0:
            raise ValueError('Number of bins `n_bins` must be positive.')
        self.r_3d = r_3d
        self.n_bins = n_bins

    def r_limit(self, i):
        '''Calculates the limit for bins of constant volume.'''
        if i < 0 or i > self.n_bins:
            raise IndexError('Index `i` must be between 0 and `n_bins`.')
        return self.r_3d * np.sqrt(i / self.n_bins)

In [ ]:
r_3d = 1000  # Radius of the simulation in Euclidean space [Mpc]
d_4d = 105   # Diameter of the simulation in 4D space [Mpc]
n_bins = 224

In [ ]:
last_cell_size = n_bins*np.pi / (2*np.arctan(r_3d/d_4d)) - n_bins
log.info(f'last_cell_size = {last_cell_size}')

binner_comg = CylindricalConstantRadius(r_3d=r_3d, n_bins=n_bins)
log.info(f'binner_comg.bin_width = {binner_comg.bin_width}')
log.info(f'binner_comg.bin_width * n_bins = {binner_comg.bin_width * n_bins}')
# binner_cvol = CylindricalConstantVolume(r_3d=r_3d, d_4d=d_4d, n_bins=n_bins)
# log.info(f'binner_cvol.omega_max = {binner_cvol.omega_max}')
# log.info(f'binner_cvol.bin_width = {binner_cvol.bin_width}')
# log.info(f'binner_cvol.bin_width * n_bins = {binner_cvol.bin_width * n_bins}')